# Adivina el animal

En esta práctica implementaremos el ejemplo de sistema experto visto en la teoría. A partir de unas características, el sistema deberá adivinar de qué animal se trata. Solo tendrá en cuenta los animales que aparecen en el esquema de teoría, que se muestra en la imagen siguiente:

<br />
<div>
<img src="https://lawer.github.io/mia/apunts/images%2FAND-OR-Tree.png" width="900"/>
</div>

Utilizaremos el módulo de Python `experta` para implementar un sistema experto con **encadenamiento hacia delante** (*forward chaining*). Este módulo simplifica mucho la creación de un sistema experto: basta con definir las reglas y los hechos que utilizará.

Si implementáramos nuestro propio motor de inferencia, podríamos tener problemas de rendimiento cuando el sistema tuviera muchas reglas, porque habría que comprobar todas las reglas para cada hecho. En cambio, `experta` utiliza una implementación optimizada (basada en el algoritmo RETE), que permite trabajar con sistemas expertos con muchas reglas y hechos.

A continuación, instalaremos e importaremos las bibliotecas necesarias para la práctica.

In [19]:
!pip install git+https://github.com/openmotics/om-experta.git

  Cloning https://github.com/openmotics/om-experta.git to /private/var/folders/zm/06zm__c5637bs4z8mhp4pfph0000gn/T/pip-req-build-b_agizuu
  Running command git clone --filter=blob:none --quiet https://github.com/openmotics/om-experta.git /private/var/folders/zm/06zm__c5637bs4z8mhp4pfph0000gn/T/pip-req-build-b_agizuu
  Resolved https://github.com/openmotics/om-experta.git to commit d35d53708a46482e1ee4e3a4bc1a36bc03492913
  Preparing metadata (setup.py) ... done

[notice] A new release of pip available: 22.3.1 -> 23.3.2
[notice] To update, run: pip install --upgrade pip


In [20]:
from experta import *

## Definición del sistema experto

Definiremos el sistema experto como una clase que hereda de `KnowledgeEngine`. Cada regla se define mediante una función decorada con `@Rule`, que especifica cuándo debe ejecutarse. Dentro de la regla podemos añadir hechos con la función `declare`; esos hechos pueden activar otras reglas mediante el motor de inferencia.

El contenido de `@Rule` es una expresión lógica cuyo resultado es `True` o `False`. Puede incluir **hechos**, operadores lógicos (`AND`, `OR`, `NOT`) y operadores de comparación (`==`, `!=`, `<`, `>`, `<=`, `>=`). Los hechos son objetos de la clase `Fact` y pueden tener atributos, que se indican como parámetros. Por ejemplo, `Fact('mamífer', pel=True)` tiene el atributo `pelo` con valor `True`.

También podemos utilizar `MATCH` para indicar que un atributo puede tener cualquier valor. Por ejemplo, `Fact('mamífer', pel=MATCH.pel)` representa un hecho cuyo atributo `pelo` puede tomar cualquier valor.

In [21]:
class Animals(KnowledgeEngine):
    # Regles    
    @Rule(OR(
        AND(Fact('dents afilades'), Fact('ungles'), Fact('ulls mirant endavant')),
        Fact('carnivor')))
    # Si l'animal té dents afilades, ungles i ulls mirant endavant, o menja carn, llavors és un carnivor
    def carnivor(self):
        # Afegim el fet 'carnivor' al sistema expert
        self.declare(Fact('carnivor'))

    @Rule(OR(Fact('pel'), Fact('dona llet')))
    # Si l'animal té pèl o dóna llet, llavors és un mamífer
    def mamifer(self):
        self.declare(Fact('mamifer'))

    @Rule(Fact('mamifer'),
          OR(Fact('te peulles'), Fact('mastega el bolus')))
    # Si l'animal és un mamifer i té peülles o mastega el _bolus_, llavors es un ungulat
    def peulles(self):
        self.declare('ungulat')

    @Rule(OR(Fact('plomes'), AND(Fact('vola'), Fact('pon ous'))))
    # Si l'animal té plomes o vola i pon ous, llavors es un ocell
    def ocell(self):
        self.declare('ocell')

    @Rule(Fact('mamifer'), Fact('carnivor'),
          Fact(color='marro-vermellos'),
          Fact(pattern='taques fosques'))
    # Si l'animal és un mamifer carnivor, de color marró vermellós i amb taques fosques, llavors és un mico
    def monkey(self):
        self.declare(Fact(animal='mico'))

    @Rule(Fact('mamifer'), Fact('carnivor'),
          Fact(color='marro-vermellos'),
          Fact(pattern='ratlles fosques'))
    # Si l'animal es un mamifer carnivor, de color marró vermellós i amb ratlles fosques, llavors és un tigre
    def tigre(self):
        self.declare(Fact(animal='tigre'))

    @Rule(Fact('ungulat'),
          Fact('coll llarg'),
          Fact('long legs'),
          Fact(pattern='taques fosques'))
    # Si l'animal és un ungulat, de coll llarg, potes llargues i amb taques fosques, llavors és una girafa
    def girafa(self):
        self.declare(Fact(animal='girafa'))

    @Rule(Fact('ungulat'),
          Fact(pattern='ratlles fosques'))
    # Si l'animal es un ungulat amb ratlles fosques, llavors es una zebra
    def zebra(self):
        self.declare(Fact(animal='zebra'))

    @Rule(Fact('ocell'),
          Fact('coll llarg'),
          Fact('no vola'),
          Fact(color='blanc i negre'))
    # Si l'animal es un ocell, de coll llarg, no vola i de color blanc i negre, llavors és un estruç
    def estruc(self):
        self.declare(Fact(animal='estruc'))

    @Rule(Fact('ocell'),
          Fact('nada'),
          Fact('no vola'),
          Fact(color='blanc i negre'))
    # Si l'animal es un ocell, nada, no vola i de color blanc i negre, llavors es un pingüí
    def pingui(self):
        self.declare(Fact(animal='pingui'))

    @Rule(Fact('ocell'),
          Fact('vola be'))
    # Si l'animal es un ocell i vola bé, llavors es un albatros
    def albatros(self):
        self.declare(Fact(animal='albatros'))

    @Rule(Fact(animal=MATCH.a))
    # Si està definit l'animal, llavors el mostrem
    def print_result(self, a):
        print(f"L'animal és un {a}")

    def agregar_fets(self, fets):
        # Afegim els fets passats com a paràmetre (fets) al sistema expert
        # Utilitzarem aquesta funció per a inicialitzar el sistema expert
        for f in fets:
            self.declare(f)

# Prueba del sistema experto

Una vez definida la base de conocimiento, inicializamos la memoria de trabajo con unos hechos y llamamos al método `run()` para realizar la inferencia. Como resultado, se añaden nuevos hechos inferidos a la memoria de trabajo, incluido el hecho final sobre el animal, si hemos configurado correctamente todos los hechos iniciales.

Veamos un ejemplo: inicializamos el sistema experto con los hechos `pel`, `dents afilades`, `ungles` y `ulls mirant endavant`. Estos hechos bastan para inferir que el animal es carnívoro.

In [22]:
# Creem una instància del nostre sistema expert
animals = Animals()

# Reset del sistema expert. Necessari per a tornar a l'estat inicial.
animals.reset()

# Afegim els fets inicials.
animals.agregar_fets(
    [Fact('pel'), Fact('dents afilades'), Fact('ungles'), Fact('ulls mirant endavant')]
)

# Executem el sistema expert
animals.run()

animals.facts

FactList([(0, InitialFact()),
          (1, Fact('pel')),
          (2, Fact('dents afilades')),
          (3, Fact('ungles')),
          (4, Fact('ulls mirant endavant')),
          (5, Fact('carnivor')),
          (6, Fact('mamifer'))])

Podemos ver que, a partir de los hechos iniciales, se ha inferido el hecho `carnivor`; a partir del hecho `pel`, también se ha inferido `mamifer`.

Añadamos primero el hecho `marro-vermellos` y, después, `taques fosques`. Con el primer hecho todavía no se infiere ningún animal. Al añadir el segundo, se infiere `mico` a partir de `mamifer`, `carnivor`, `marro-vermellos` y `taques fosques`.

In [23]:
animals.agregar_fets(
    [Fact(color='marro-vermellos')]
)
animals.run()
animals.facts

FactList([(0, InitialFact()),
          (1, Fact('pel')),
          (2, Fact('dents afilades')),
          (3, Fact('ungles')),
          (4, Fact('ulls mirant endavant')),
          (5, Fact('carnivor')),
          (6, Fact('mamifer')),
          (7, Fact(color='marro-vermellos'))])

In [24]:
animals.agregar_fets(
    [Fact(pattern='taques fosques')]
)
animals.run()
animals.facts

L'animal és un mico


FactList([(0, InitialFact()),
          (1, Fact('pel')),
          (2, Fact('dents afilades')),
          (3, Fact('ungles')),
          (4, Fact('ulls mirant endavant')),
          (5, Fact('carnivor')),
          (6, Fact('mamifer')),
          (7, Fact(color='marro-vermellos')),
          (8, Fact(pattern='taques fosques')),
          (9, Fact(animal='mico'))])